In [ ]:
#!/usr/bin/env python3
"""Plot Shabdiz throughput and latency versus client time.

The input directory must contain one or more client_*.log files produced by
run_shabdiz_4nodes.py. Two independent figures and one parsed CSV are written
to <run_dir>/plots.
"""

from __future__ import annotations

import argparse
import math
import os
import re
from pathlib import Path

import matplotlib

if not os.environ.get("DISPLAY"):
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator
import numpy as np
import pandas as pd


DEFAULT_LATEST_RUN_FILE = (
    Path.home() / "work" / "experiments" / "shabdiz" / "latest_4node_run.txt"
)

CLIENT_SEC_PATTERN = re.compile(
    r"\[CLIENT_SEC\]\s+"
    r"sec=(?P<sec>\d+)\s+"
    r"requests=(?P<requests>\d+)\s+"
    r"throughput=(?P<throughput>[0-9.eE+-]+)\s+"
    r"avg_latency_ms=(?P<avg_latency_ms>[0-9.eE+-]+|nan)\s+"
    r"p50_latency_ms=(?P<p50_latency_ms>[0-9.eE+-]+|nan)\s+"
    r"p99_latency_ms=(?P<p99_latency_ms>[0-9.eE+-]+|nan)"
    r"(?:\s+p999_latency_ms=(?P<p999_latency_ms>[0-9.eE+-]+|nan))?"
)

LATENCY_COLUMNS = {
    "mean": ("avg_latency_ms", "Mean latency (ms)"),
    "p50": ("p50_latency_ms", "p50 latency (ms)"),
    "p99": ("p99_latency_ms", "p99 latency (ms)"),
    "p999": ("p999_latency_ms", "p99.9 latency (ms)"),
}

FIGURE_SIZE = (10, 6)
FIGURE_DPI = 160


# Parsing


def parse_float(value: str | None) -> float:
    if value is None or value.lower() == "nan":
        return float("nan")
    return float(value)


def parse_client_log(log_path: Path, client_id: int) -> pd.DataFrame:
    rows: list[dict[str, float | int]] = []

    with log_path.open(errors="ignore") as log_file:
        for line in log_file:
            match = CLIENT_SEC_PATTERN.search(line)
            if match is None:
                continue

            rows.append(
                {
                    "client_id": client_id,
                    "sec": int(match.group("sec")),
                    "requests": int(match.group("requests")),
                    "throughput_tps": float(match.group("throughput")),
                    "avg_latency_ms": parse_float(match.group("avg_latency_ms")),
                    "p50_latency_ms": parse_float(match.group("p50_latency_ms")),
                    "p99_latency_ms": parse_float(match.group("p99_latency_ms")),
                    "p999_latency_ms": parse_float(match.group("p999_latency_ms")),
                }
            )

    return pd.DataFrame(rows)


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    value_array = values.to_numpy(dtype=float)
    weight_array = weights.to_numpy(dtype=float)
    valid = np.isfinite(value_array) & np.isfinite(weight_array)

    if not valid.any() or weight_array[valid].sum() <= 0:
        return float("nan")

    return float(np.average(value_array[valid], weights=weight_array[valid]))


def aggregate_clients_by_second(client_data: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, float | int]] = []

    for second, group in client_data.groupby("sec", sort=True):
        rows.append(
            {
                "sec": int(second),
                "requests": int(group["requests"].sum()),
                "throughput_tps": float(group["throughput_tps"].sum()),
                "avg_latency_ms": weighted_mean(
                    group["avg_latency_ms"], group["requests"]
                ),
                # These combined percentiles are request-weighted approximations.
                # Exact global percentiles require raw per-request samples.
                "p50_latency_ms": weighted_mean(
                    group["p50_latency_ms"], group["requests"]
                ),
                "p99_latency_ms": weighted_mean(
                    group["p99_latency_ms"], group["requests"]
                ),
                "p999_latency_ms": weighted_mean(
                    group["p999_latency_ms"], group["requests"]
                ),
            }
        )

    return pd.DataFrame(rows).sort_values("sec").reset_index(drop=True)


def find_client_logs(run_dir: Path) -> list[Path]:
    client_logs = sorted(run_dir.glob("client_*.log"))

    if not client_logs:
        legacy_log = run_dir / "client.log"
        if legacy_log.exists():
            client_logs = [legacy_log]

    if not client_logs:
        raise FileNotFoundError(f"No client_*.log files found in {run_dir}")

    return client_logs


def load_timeseries(
    run_dir: Path,
    start_sec: int | None,
    end_sec: int | None,
) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []

    for client_id, log_path in enumerate(find_client_logs(run_dir)):
        print(f"Parsing {log_path}")
        frame = parse_client_log(log_path, client_id)
        if frame.empty:
            print(f"Warning: no [CLIENT_SEC] records found in {log_path}")
            continue
        frames.append(frame)

    if not frames:
        raise RuntimeError("No usable [CLIENT_SEC] records were found.")

    aggregate = aggregate_clients_by_second(pd.concat(frames, ignore_index=True))

    first_second = int(aggregate["sec"].min()) if start_sec is None else start_sec
    last_second = int(aggregate["sec"].max()) if end_sec is None else end_sec

    if first_second > last_second:
        raise ValueError("start-sec must be less than or equal to end-sec")

    complete_seconds = pd.DataFrame(
        {"sec": np.arange(first_second, last_second + 1, dtype=int)}
    )
    aggregate = complete_seconds.merge(aggregate, on="sec", how="left")

    aggregate["requests"] = aggregate["requests"].fillna(0).astype(int)
    aggregate["throughput_tps"] = aggregate["throughput_tps"].fillna(0.0)

    # Keep latency as NaN in seconds with no completed requests.
    return aggregate


# Plotting


def compact_number(value: float, position: int | None = None) -> str:
    del position
    if not math.isfinite(value):
        return ""
    if abs(value) >= 1_000_000:
        return f"{value / 1_000_000:.3g}M"
    if abs(value) >= 1_000:
        return f"{value / 1_000:.3g}k"
    return f"{value:.3g}"


def add_smoothed_columns(data: pd.DataFrame, window: int) -> pd.DataFrame:
    result = data.copy()
    metric_columns = [
        "throughput_tps",
        "avg_latency_ms",
        "p50_latency_ms",
        "p99_latency_ms",
        "p999_latency_ms",
    ]

    for column in metric_columns:
        result[f"{column}_plot"] = (
            result[column]
            .rolling(window=window, center=True, min_periods=1)
            .mean()
        )

    return result


def save_figure(fig: plt.Figure, output_stem: Path) -> None:
    fig.tight_layout()
    fig.savefig(output_stem.with_suffix(".pdf"), dpi=FIGURE_DPI, bbox_inches="tight")
    fig.savefig(output_stem.with_suffix(".png"), dpi=FIGURE_DPI, bbox_inches="tight")
    print(f"Saved {output_stem.with_suffix('.pdf')}")
    print(f"Saved {output_stem.with_suffix('.png')}")


def plot_throughput(data: pd.DataFrame, output_dir: Path) -> plt.Figure:
    fig, axis = plt.subplots(figsize=FIGURE_SIZE)
    axis.plot(
        data["sec"],
        data["throughput_tps_plot"],
        linewidth=2,
        label="SHABDIZ",
    )
    axis.set_xlabel("Time since client start (s)")
    axis.set_ylabel("Throughput (txns/s)")
    axis.set_xlim(int(data["sec"].min()), int(data["sec"].max()))
    axis.yaxis.set_major_locator(MaxNLocator(nbins=8))
    axis.yaxis.set_major_formatter(FuncFormatter(compact_number))
    axis.legend(frameon=False)
    save_figure(fig, output_dir / "throughput_vs_time")
    return fig


def plot_latency(
    data: pd.DataFrame,
    output_dir: Path,
    latency_metric: str,
) -> plt.Figure:
    latency_column, latency_label = LATENCY_COLUMNS[latency_metric]

    fig, axis = plt.subplots(figsize=FIGURE_SIZE)
    axis.plot(
        data["sec"],
        data[f"{latency_column}_plot"],
        linewidth=2,
        label="SHABDIZ",
    )
    axis.set_xlabel("Time since client start (s)")
    axis.set_ylabel(latency_label)
    axis.set_xlim(int(data["sec"].min()), int(data["sec"].max()))
    axis.yaxis.set_major_locator(MaxNLocator(nbins=8))
    axis.legend(frameon=False)
    save_figure(fig, output_dir / f"{latency_metric}_latency_vs_time")
    return fig


# CLI


def resolve_run_dir(argument: str | None) -> Path:
    if argument is not None:
        run_dir = Path(argument).expanduser().resolve()
    else:
        if not DEFAULT_LATEST_RUN_FILE.exists():
            raise FileNotFoundError(
                "No run directory was supplied and the latest-run marker does not exist: "
                f"{DEFAULT_LATEST_RUN_FILE}"
            )
        run_dir = Path(DEFAULT_LATEST_RUN_FILE.read_text().strip()).expanduser()

    if not run_dir.is_dir():
        raise NotADirectoryError(f"Run directory does not exist: {run_dir}")
    return run_dir


def parse_arguments() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Plot Shabdiz throughput and latency versus time."
    )
    parser.add_argument(
        "run_dir",
        nargs="?",
        help=(
            "Experiment result directory. When omitted, the script reads "
            f"{DEFAULT_LATEST_RUN_FILE}."
        ),
    )
    parser.add_argument(
        "--start-sec",
        type=int,
        default=None,
        help="First client second to include. Default: first parsed second.",
    )
    parser.add_argument(
        "--end-sec",
        type=int,
        default=None,
        help="Last client second to include. Default: last parsed second.",
    )
    parser.add_argument(
        "--rolling-window",
        type=int,
        default=3,
        help="Centered moving-average window in seconds. Default: 3.",
    )
    parser.add_argument(
        "--latency",
        choices=sorted(LATENCY_COLUMNS),
        default="mean",
        help="Latency series to plot. Default: mean.",
    )
    parser.add_argument(
        "--show",
        action="store_true",
        help="Display figures interactively after saving them.",
    )
    return parser.parse_args()


def main() -> None:
    args = parse_arguments()

    if args.rolling_window < 1:
        raise ValueError("rolling-window must be at least 1")

    run_dir = resolve_run_dir(args.run_dir)
    output_dir = run_dir / "plots"
    output_dir.mkdir(parents=True, exist_ok=True)

    data = load_timeseries(run_dir, args.start_sec, args.end_sec)
    data = add_smoothed_columns(data, args.rolling_window)

    csv_path = output_dir / "shabdiz_timeseries.csv"
    data.to_csv(csv_path, index=False)
    print(f"Saved {csv_path}")

    nonzero = data[data["requests"] > 0]
    print("\nSelected-window summary")
    print(f"  Seconds: {int(data['sec'].min())} through {int(data['sec'].max())}")
    print(f"  Completed requests: {int(data['requests'].sum())}")
    print(f"  Mean throughput: {data['throughput_tps'].mean():.2f} txns/s")
    if nonzero.empty:
        print("  Request-weighted mean latency: unavailable")
    else:
        mean_latency = weighted_mean(
            nonzero["avg_latency_ms"], nonzero["requests"]
        )
        print(f"  Request-weighted mean latency: {mean_latency:.2f} ms")

    throughput_figure = plot_throughput(data, output_dir)
    latency_figure = plot_latency(data, output_dir, args.latency)

    if args.show:
        plt.show()
    else:
        plt.close(throughput_figure)
        plt.close(latency_figure)


if __name__ == "__main__":
    main()